# Phase and Peak Fitting

Use the structure-agnostic `PeakFitPlan` path for a compact pilot fit.
Structure-informed phase fitting remains a separate operation because it
needs explicit CIF-backed phase models and scientific constraints.


In [ ]:
import numpy as np
import ipywidgets as widgets
from IPython.display import clear_output, display

from xrd_tools.analysis import AnalysisInput, PeakFitAnalyzer, PeakFitPlan
from xrd_tools.gui.widgets import PatternViewer, PeakFitControls
from xrd_tools.viz import plot_peak_fit


In [ ]:
import os
from pathlib import Path

# Smoke mode is deterministic and has no external-data requirement.
# Set XDART_NOTEBOOK_SMOKE=0 and XDART_TEST_DATA=/path/to/data for real data.
SMOKE_MODE = os.environ.get("XDART_NOTEBOOK_SMOKE", "1") != "0"
TEST_DATA = Path(os.environ.get("XDART_TEST_DATA", "")).expanduser()


In [ ]:
q = np.linspace(2.3, 3.2, 320)
intensity = 10 + 120 * np.exp(-0.5 * ((q - 2.76) / 0.018) ** 2)
output = widgets.Output()
status = widgets.HTML("<i>Change fit settings, then use the shared Fit action.</i>")
NOTEBOOK_STATE = {"runs": 0, "outcome": None}

def run_pilot(params=None):
    params = params or controls.get_params()
    with output:
        clear_output(wait=True)
        try:
            positions = tuple(params["positions"] or (2.76,))
            plan = PeakFitPlan(positions=positions, model=params["model"], background=params["background"] if params["background"] in {"none", "constant", "linear"} else "linear", sigma_init=params["sigma_init"], sigma_bounds=params["sigma_bounds"], center_bounds_delta=params["center_bounds_delta"], fit_kwargs={"method": "leastsq"})
            outcome = PeakFitAnalyzer(plan).analyze(AnalysisInput(label="pilot", x=q, y=intensity, x_unit="q_A^-1"))
            assert outcome.ok, outcome.message
            display(plot_peak_fit(q, intensity, outcome.result.payload, title="Pilot peak fit"))
            NOTEBOOK_STATE.update(runs=NOTEBOOK_STATE["runs"] + 1, outcome=outcome, plan=plan)
            status.value = "<b>Pilot fit complete.</b>"
        except Exception as exc:
            status.value = f"<b>Pilot fit failed:</b> {exc}"
            raise

controls = PeakFitControls(on_fit=run_pilot)
controls.n_peaks.value = 1
controls.peak_positions.value = "2.76"
controls.peak_model.value = "gaussian"
controls.bg_model.value = "linear"
controls.sigma_init.value = 0.02
controls.q_min.value, controls.q_max.value = 2.3, 3.2
viewer = PatternViewer(patterns=[(q, intensity, "pilot")])
display(widgets.VBox([controls.widget, viewer.widget, status, output]))
NOTEBOOK_ACTIONS = {"run_pilot": run_pilot}
if SMOKE_MODE or os.environ.get("XDART_NOTEBOOK_AUTORUN") == "1":
    run_pilot()
